# 01 — Ingest & Parse

**Audience:** Engineers following the technical walkthrough.

**What this shows:** Download corpus → upload to blob → Azure Document Intelligence Layout API → parsed JSON in `data/parsed/`.

**Prerequisites:** Azure credentials in `.env`, venv activated. (No prior notebook dependency.)

# 01 — Ingest & Parse

Upload the Phase 1 corpus (public IRS PDFs) to Blob Storage and parse each
with Azure AI Document Intelligence **Layout** (Markdown output).

**Auth:** Entra ID via `DefaultAzureCredential` (your `az login`). No
keys. Requires `Storage Blob Data Contributor` on the storage account
and `Cognitive Services User` on the Document Intelligence resource.

**Outputs** (per doc, under `data/parsed/<document_id>/`):

- `layout.json` — raw DI AnalyzeResult (paragraphs, pages, tables, spans).
- `document.md` — DI Markdown rendering, heading-aware.
- `meta.json` — normalized record (page count, sha, blob URL).

This notebook walks through **one** document for inspection. Use
`scripts/ingest.py` to run the whole corpus non-interactively.

In [ ]:
from pathlib import Path
import json, sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT))

from sentcite.config import AzureConfig
from sentcite.ingest import ingest_and_parse
from scripts.download_corpus import CORPUS

cfg = AzureConfig.from_env()
DOC_ID = 'irs_pub_583'
doc = next(d for d in CORPUS if d.document_id == DOC_ID)
pdf = REPO_ROOT / 'data' / 'raw_pdfs' / doc.filename
pdf, pdf.exists()

In [ ]:
results = ingest_and_parse(
    [(DOC_ID, pdf)],
    parsed_dir=REPO_ROOT / 'data' / 'parsed',
    cfg=cfg,
)
r = results[0]
{
    'document_id': r.document_id,
    'page_count': r.page_count,
    'blob_url': r.blob_url,
    'layout_json_path': r.layout_json_path,
    'markdown_path': r.markdown_path,
}

In [ ]:
# Peek at the first ~1 KB of the Markdown rendering.
print(Path(r.markdown_path).read_text()[:1200])

In [ ]:
# Summary stats from the raw AnalyzeResult. Paragraph + page counts are
# what chunking.py consumes next.
layout = json.loads(Path(r.layout_json_path).read_text())
paragraphs = layout.get('paragraphs', [])
headings = [p for p in paragraphs if (p.get('role') or '').startswith('section') or p.get('role') == 'title']
{
    'pages': len(layout.get('pages', [])),
    'paragraphs': len(paragraphs),
    'headings_detected': len(headings),
    'tables': len(layout.get('tables', [])),
    'figures': len(layout.get('figures', [])),
}

## What's next

👉 **Next step:** [`02_chunk_and_index.ipynb`](02_chunk_and_index.ipynb)  
We walk the parsed docs into sentence-aware chunks and build the dual-layout Azure AI Search indexes.